# MARBERT reproducibility audit: run everything on Google Colab

This notebook runs the full pipeline: data, then 36 fine-tuning runs, then the integrity check, then every table and figure.

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. Run the cells from top to bottom. Finished runs are skipped, so after a disconnect just run the cells again.
3. Run outputs and checkpoints are kept on your Google Drive, so a disconnect loses at most one epoch.

Rough time on a T4: AJGT runs take a few minutes each, ArSarcasm runs well under an hour each, and ASTD/CAMelSent runs are the longest. Split the runs across sessions or teammates with the `--datasets` / `--protocols` options in step 5.

In [ ]:
!nvidia-smi

## 1. Mount Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/zayedd2/marbert-repro-audit.git"   # <- change to the group's repository
DRIVE_DIR = "/content/drive/MyDrive/marbert-repro-audit"            # run outputs + checkpoints live here
!git clone $REPO_URL /content/marbert-repro-audit
%cd /content/marbert-repro-audit
!pip install -q -r requirements.txt
!python -m pytest -q

## 2. Keep run outputs on Drive
`results/runs` becomes a link to Drive, so completed runs survive a disconnect.

In [ ]:
!mkdir -p $DRIVE_DIR/runs $DRIVE_DIR/checkpoints
!rm -rf results/runs && ln -s $DRIVE_DIR/runs results/runs
!ls -la results

## 3. Data
ArSarcasm and AJGT download automatically. For ASTD/CAMelSent, upload the per-source files to `data/raw/camelsent/` and create `data/camelsent_sources.yaml` first (see `data/README.md`).

In [ ]:
!python scripts/prepare_data.py arsarcasm
!python scripts/prepare_data.py ajgt
# !python scripts/prepare_data.py astd --manifest data/camelsent_sources.yaml
!python scripts/data_audit.py

## 4. Check the plan
This lists every run and whether it is already done.

In [ ]:
!python scripts/run_experiments.py --list

## 4b. Smoke test on the GPU (about 2 minutes)
One epoch of one AJGT run into a scratch folder. It proves that downloads, tokenisation, training and saving work before you spend hours. Nothing is written to `results/`.

In [ ]:
!python scripts/run_experiments.py --datasets ajgt --models marbert arabert --seeds 42 --protocols independent --epochs 1 --results-root /tmp/smoke
!cat /tmp/smoke/runs/independent/ajgt/marbert/seed42/epoch_metrics.csv

## 5. Fine-tune
Run one line at a time if the session is short. Each line can be re-run safely.

In [ ]:
CK = f"--resume-root {DRIVE_DIR}/checkpoints"
!python scripts/run_experiments.py --datasets ajgt {CK}
!python scripts/run_experiments.py --datasets arsarcasm {CK}
!python scripts/run_experiments.py --datasets astd {CK}

## 6. Verify, then build every table and figure

In [ ]:
!python scripts/verify_results.py
!python scripts/make_report_assets.py
!pip freeze > requirements-lock.txt
from IPython.display import Markdown, display
display(Markdown(open('results/SUMMARY.md').read()))

## 7. Save the results into the repository
Replace the Drive link with a real copy, then download a zip to commit (or push directly with a GitHub token).

In [ ]:
!rm results/runs && cp -r $DRIVE_DIR/runs results/runs
!zip -qr results_for_repo.zip results report/generated requirements-lock.txt data/splits data/MANIFEST.json -x "results/analysis/*"
from google.colab import files
files.download('results_for_repo.zip')